## Sampling over the glacier Centroid (Rep Pt instead of cenlat and cenlon)

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

In [21]:
# -----------------------------
# Settings
# -----------------------------
nc_path = Path("w5e5aggregrates/W5E5_calyr_aggregates.nc")
vars_to_sample = ["tas_mean", "tas_median", "tas_min","tas_max","tas_std", "rsds_mean","rsds_median","rsds_min","rsds_max","rsds_std", "pr_sum"]

In [22]:
# -----------------------------
# Load dataset
# -----------------------------
ds = xr.open_dataset(nc_path, chunks={})  # add chunks={'year':..., 'lat':..., 'lon':...} if big

In [23]:
# If time is used instead of year:
if "time" in ds.dims and "year" not in ds.dims:
    print('Fixing times')
    ds = ds.assign_coords(year=ds["time"].dt.year).swap_dims({"time": "year"}).drop_vars("time")

# Ensure lon convention matches your shapefile longitudes:
# If ds lon is 0..360 but shapes are -180..180, convert ds
if ds["lon"].max() > 180:
    print('Fixing lons')
    ds = ds.assign_coords(lon=(((ds["lon"] + 180) % 360) - 180)).sortby("lon")

Fixing times


In [ ]:
rgi_root = Path("rgi")
out_dir = Path("pergla_annualmetw5e5_sampled_csv")
out_dir.mkdir(parents=True, exist_ok=True)

In [25]:
# -----------------------------
# Process each region shapefile
# -----------------------------
# Adjust this glob to match your directory structure.
# Many RGI folders have shapefiles like *glaciers*.shp
shp_paths = sorted(rgi_root.rglob("*.shp"))

for shp_path in shp_paths:
    # Skip misc folders if needed
    # if "misc" in shp_path.parts: continue

    gdf = gpd.read_file(shp_path)

    # Ensure geometry is in lon/lat (WGS84)
    if gdf.crs is None:
        raise ValueError(f"{shp_path} has no CRS; set it before proceeding.")
    gdf = gdf.to_crs("EPSG:4326")

    id_col = 'RGIId'

    # Use representative_point() instead of centroid if you want a point guaranteed inside polygon
    rep_pts = gdf.geometry.representative_point()
    lats = xr.DataArray(rep_pts.y.to_numpy(), dims="glacier")
    lons = xr.DataArray(rep_pts.x.to_numpy(), dims="glacier")

    # Sample: nearest grid cell (fast). For bilinear, use .interp(...)
    sampled = ds[vars_to_sample].sel(lat=lats, lon=lons, method="nearest")
    # Bring glacier IDs + attributes in
    attrs = gdf[[id_col, "GLIMSId","CenLon", "CenLat","Slope", "Area", "Aspect", "Zmed", "Zmin", "Zmax"]].copy()
    attrs = attrs.rename(columns={id_col: "rgi_id"})

    # Convert sampled to tidy dataframe: (glacier, year) rows
    df_vals = sampled.to_dataframe().reset_index()  # columns: glacier, year, variables...
    df_vals["rgi_id"] = attrs["rgi_id"].iloc[df_vals["glacier"].to_numpy()].to_numpy()

    # Join the static glacier attrs
    df = df_vals.merge(attrs, on="rgi_id", how="left")

    # Keep columns ordered
    col_order = ["rgi_id", "year", "GLIMSId","CenLon", "CenLat", "Slope", "Area", "Aspect", "Zmed", "Zmin", "Zmax"] + vars_to_sample
    df = df[col_order].sort_values(["rgi_id", "year"])

    # Write per-shapefile (often per region)
    out_csv = out_dir / f"{shp_path.stem}_met_sampled.csv"
    df.to_csv(out_csv, index=False)
    print(f"Wrote {out_csv}  ({len(df):,} rows)")

Wrote pergla_sampled_csv/01_rgi60_Alaska_met_sampled.csv  (1,111,428 rows)
Wrote pergla_sampled_csv/02_rgi60_WesternCanadaUS_met_sampled.csv  (773,055 rows)
Wrote pergla_sampled_csv/03_rgi60_ArcticCanadaNorth_met_sampled.csv  (186,796 rows)
Wrote pergla_sampled_csv/04_rgi60_ArcticCanadaSouth_met_sampled.csv  (304,015 rows)
Wrote pergla_sampled_csv/05_rgi60_GreenlandPeriphery_met_sampled.csv  (830,701 rows)
Wrote pergla_sampled_csv/06_rgi60_Iceland_met_sampled.csv  (23,288 rows)
Wrote pergla_sampled_csv/07_rgi60_Svalbard_met_sampled.csv  (66,215 rows)
Wrote pergla_sampled_csv/08_rgi60_Scandinavia_met_sampled.csv  (140,097 rows)
Wrote pergla_sampled_csv/09_rgi60_RussianArctic_met_sampled.csv  (43,829 rows)
Wrote pergla_sampled_csv/10_rgi60_NorthAsia_met_sampled.csv  (211,191 rows)
Wrote pergla_sampled_csv/11_rgi60_CentralEurope_met_sampled.csv  (161,007 rows)
Wrote pergla_sampled_csv/12_rgi60_CaucasusMiddleEast_met_sampled.csv  (77,408 rows)
Wrote pergla_sampled_csv/13_rgi60_CentralAsia_

## Aggregation over the Glacier

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
from rasterio import features
from affine import Affine
from pathlib import Path


In [2]:
# -----------------------------
# Settings
# -----------------------------
nc_path = Path("w5e5aggregrates/W5E5_calyr_aggregates.nc")
vars_to_sample = ["tas_mean", "tas_median", "tas_min","tas_max","tas_std", "rsds_mean","rsds_median","rsds_min","rsds_max","rsds_std", "pr_sum"]

In [3]:
# -----------------------------
# Load dataset
# -----------------------------
ds = xr.open_dataset(nc_path, chunks={})  # add chunks={'year':..., 'lat':..., 'lon':...} if big

In [4]:
# If time is used instead of year:
if "time" in ds.dims and "year" not in ds.dims:
    print('Fixing times')
    ds = ds.assign_coords(year=ds["time"].dt.year).swap_dims({"time": "year"}).drop_vars("time")

# Ensure lon convention matches your shapefile longitudes:
# If ds lon is 0..360 but shapes are -180..180, convert ds
if ds["lon"].max() > 180:
    print('Fixing lons')
    ds = ds.assign_coords(lon=(((ds["lon"] + 180) % 360) - 180)).sortby("lon")

Fixing times


In [5]:
lon = ds["lon"].values
lat = ds["lat"].values

dlon = np.mean(np.diff(lon))
dlat = np.mean(np.diff(lat))

transform = Affine(
    dlon, 0.0, lon.min() - dlon / 2,
    0.0, -abs(dlat), lat.max() + abs(dlat) / 2
)

In [6]:
rgi_root = Path("rgi")
out_dir = Path("pergla__annualmetw5e5_agg_csv")
out_dir.mkdir(parents=True, exist_ok=True)

In [8]:
shp_paths = sorted(rgi_root.rglob("*.shp"))
for shp_path in shp_paths[5:6]:
    # Skip misc folders if needed
    # if "misc" in shp_path.parts: continue

    gdf = gpd.read_file(shp_path)

    # Ensure geometry is in lon/lat (WGS84)
    if gdf.crs is None:
        raise ValueError(f"{shp_path} has no CRS; set it before proceeding.")
    gdf = gdf.to_crs("EPSG:4326")

    # Integer label per glacier (numerical form of rgi_id)
    gdf["glac_label"] = np.arange(1, len(gdf) + 1, dtype=np.int32)

    # Rasterize glaciers allowing for all pixels intersecting a glacier polygon to have the value ofthe glacier label (numerical form of rgi_id)
    label_arr = features.rasterize(
        shapes=zip(gdf.geometry, gdf["glac_label"]),
        out_shape=(ds.dims["lat"], ds.dims["lon"]),
        transform=transform,
        fill=0,
        dtype="int32",
        all_touched=True
    )

    labels = xr.DataArray(
        label_arr,
        coords={"lat": ds.lat, "lon": ds.lon},
        dims=("lat", "lon")
    )

    met = ds[vars_to_sample].where(labels > 0) # Picks only where the glaciers exist (as labels are from 1 to num of glaciers)

    # Zonal stats (robust)
    met_stacked = met.stack(z=("lat", "lon"))
    lab_stacked = labels.stack(z=("lat", "lon"))

    gb = met_stacked.groupby(lab_stacked)

    mean_da = gb.mean(dim="z", skipna=True)
    med_da  = gb.median(dim="z", skipna=True)
    min_da  = gb.min(dim="z", skipna=True)
    max_da  = gb.max(dim="z", skipna=True)
    std_da  = gb.std(dim="z", skipna=True)

    # rename the group coordinate to a stable name
    # (xarray often names it "group"; sometimes it uses the DataArray name)
    group_coord = [c for c in mean_da.coords if c not in ("year",) and c in mean_da.dims][0]
    mean_da = mean_da.rename({group_coord: "glac_label"})
    med_da  = med_da.rename({group_coord: "glac_label"})
    min_da  = min_da.rename({group_coord: "glac_label"})
    max_da  = max_da.rename({group_coord: "glac_label"})
    std_da  = std_da.rename({group_coord: "glac_label"})

    def tidy(da, suffix):
        df = da.to_dataframe().reset_index()
        return df.rename(columns={v: f"{v}_{suffix}" for v in vars_to_sample})

    df = (tidy(mean_da, "mean")
        .merge(tidy(med_da, "med"), on=["glac_label", "year"])
        .merge(tidy(min_da, "min"), on=["glac_label", "year"])
        .merge(tidy(max_da, "max"), on=["glac_label", "year"])
        .merge(tidy(std_da, "std"), on=["glac_label", "year"]))

    # Attach glacier attributes
    attrs = gdf[[
        "RGIId", "GLIMSId", "CenLon", "CenLat",
        "Slope", "Area", "Aspect", "Zmed", "Zmin", "Zmax",
        "glac_label"
    ]].rename(columns={"RGIId": "rgi_id"})

    df = df.merge(attrs, on="glac_label", how="left").drop(columns=["glac_label"])
    df = df.sort_values(["rgi_id", "year"])

    out_csv = out_dir / f"{shp_path.stem}_met_zonal.csv"
    df.to_csv(out_csv, index=False)

    print(f"Wrote {out_csv} ({len(df):,} rows)")

/local/user/1483801751/ipykernel_148037/1873042313.py:19: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  out_shape=(ds.dims["lat"], ds.dims["lon"]),


Wrote pergla__annualmetw5e5_agg_csv/06_rgi60_Iceland_met_zonal.csv (1,804 rows)


In [9]:
met_stacked

<xarray.Dataset> Size: 474MB
Dimensions:      (year: 41, z: 259200)
Coordinates:
  * year         (year) int64 328B 1979 1980 1981 1982 ... 2016 2017 2018 2019
  * z            (z) object 2MB MultiIndex
  * lat          (z) float64 2MB 89.75 89.75 89.75 ... -89.75 -89.75 -89.75
  * lon          (z) float64 2MB -179.8 -179.2 -178.8 ... 178.8 179.2 179.8
Data variables:
    tas_mean     (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    tas_median   (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    tas_min      (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    tas_max      (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    tas_std      (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    rsds_mean    (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    rsds_median  (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    rsds_min     (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    rsds_max     (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    rsds_std     (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>
    pr_sum       (year, z) float32 43MB dask.array<chunksize=(41, 259200), meta=np.ndarray>

In [10]:
mean_da

<xarray.Dataset> Size: 80kB
Dimensions:      (glac_label: 44, year: 41)
Coordinates:
  * glac_label   (glac_label) int32 176B 0 143 148 167 187 ... 565 566 567 568
  * year         (year) int64 328B 1979 1980 1981 1982 ... 2016 2017 2018 2019
Data variables:
    tas_mean     (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    tas_median   (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    tas_min      (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    tas_max      (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    tas_std      (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    rsds_mean    (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    rsds_median  (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    rsds_min     (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    rsds_max     (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    rsds_std     (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>
    pr_sum       (glac_label, year) float32 7kB dask.array<chunksize=(1, 41), meta=np.ndarray>

In [11]:
print("DS lon:", float(ds.lon.min()), float(ds.lon.max()))
print("DS lat:", float(ds.lat.min()), float(ds.lat.max()))
print("GDF bounds:", gdf.total_bounds)

DS lon: -179.75 179.75
DS lat: -89.75 89.75
GDF bounds: [-23.82500249  63.5266627  -13.91142199  66.36759974]


## Monthly Sampling

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

In [2]:
nc_path = Path("w5e5aggregrates/W5E5_monthly_aggregates.nc")
ds = xr.open_dataset(nc_path, chunks={})
ds

<xarray.Dataset> Size: 6GB
Dimensions:           (time: 492, lat: 360, lon: 720)
Coordinates:
  * time              (time) datetime64[ns] 4kB 1979-01-01 ... 2019-12-01
  * lat               (lat) float64 3kB 89.75 89.25 88.75 ... -89.25 -89.75
  * lon               (lon) float64 6kB -179.8 -179.2 -178.8 ... 179.2 179.8
Data variables:
    tas_mean          (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    tas_min           (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    tas_max           (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    tas_median        (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    tas_std           (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    rsds_mean         (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    rsds_min          (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    rsds_max          (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    rsds_median       (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    rsds_std          (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>
    pr_monthly_total  (time, lat, lon) float32 510MB dask.array<chunksize=(492, 360, 720), meta=np.ndarray>

In [6]:
vars_to_sample = ["tas_mean", "tas_median", "tas_min","tas_max","tas_std", "rsds_mean","rsds_median","rsds_min","rsds_max","rsds_std", "pr_monthly_total"]

In [3]:
# -----------------------------
# Load dataset
# -----------------------------
ds = xr.open_dataset(nc_path, chunks={})  # add chunks={'year':..., 'lat':..., 'lon':...} if big

In [4]:
rgi_root = Path("rgi")
out_dir = Path("pergla_monthlymetw5e5_sampled_csv")
out_dir.mkdir(parents=True, exist_ok=True)

In [13]:
# -----------------------------
# Process each region shapefile
# -----------------------------
# Adjust this glob to match your directory structure.
# Many RGI folders have shapefiles like *glaciers*.shp
shp_paths = sorted(rgi_root.rglob("*.shp"))

for shp_path in shp_paths:
    # Skip misc folders if needed
    # if "misc" in shp_path.parts: continue

    gdf = gpd.read_file(shp_path)

    # Ensure geometry is in lon/lat (WGS84)
    if gdf.crs is None:
        raise ValueError(f"{shp_path} has no CRS; set it before proceeding.")
    gdf = gdf.to_crs("EPSG:4326")

    id_col = 'RGIId'
    # Bring glacier IDs + attributes in
    attrs = gdf[[id_col, "GLIMSId","CenLon", "CenLat","Slope", "Area", "Aspect", "Zmed", "Zmin", "Zmax"]].copy()
    attrs = attrs.rename(columns={id_col: "rgi_id"})

    # Use representative_point() instead of centroid if you want a point guaranteed inside polygon
    rep_pts = gdf.geometry.representative_point()
    lats = xr.DataArray(rep_pts.y.to_numpy(), dims="glacier")
    lons = xr.DataArray(rep_pts.x.to_numpy(), dims="glacier")

    # Sample: nearest grid cell (fast). For bilinear, use .interp(...)
    sampled = ds[vars_to_sample].sel(lat=lats, lon=lons, method="nearest")
    

    # Convert sampled to tidy dataframe: (glacier, year) rows
    dfm = sampled.to_dataframe().reset_index()  # columns: glacier, year, month variables..
    dfm["year"] = dfm["time"].dt.year
    dfm["month"] = dfm["time"].dt.month
    dfm["rgi_id"] = attrs["rgi_id"].iloc[dfm["glacier"].to_numpy()].to_numpy()

    # Pivot so months become columns, one row per glacier-year
    wide = dfm.pivot_table(
        index=["rgi_id", "year"],
        columns="month",
        values=vars_to_sample,
        aggfunc="first",   
    )

    # Flatten columns to var_01, var_02, ...
    wide.columns = [f"{var}_{m:02d}" for (var, m) in wide.columns]
    wide = wide.reset_index()

    # Join the static glacier attrs
    df = wide.merge(attrs, on="rgi_id", how="left")

    # Column ordering
    static_cols = ["rgi_id", "year", "GLIMSId","CenLon", "CenLat", "Slope", "Area",
                "Aspect", "Zmed", "Zmin", "Zmax"]

    month_cols = []
    for v in vars_to_sample:
        month_cols += [f"{v}_{m:02d}" for m in range(1, 13)]

    df = df[static_cols + month_cols].sort_values(["rgi_id", "year"])

    # Write per-shapefile (often per region)
    out_csv = out_dir / f"{shp_path.stem}_met_sampled.csv"
    df.to_csv(out_csv, index=False)
    print(f"Wrote {out_csv}  ({len(df):,} rows)")

Wrote pergla_monthlymetw5e5_sampled_csv/01_rgi60_Alaska_met_sampled.csv  (1,111,428 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/02_rgi60_WesternCanadaUS_met_sampled.csv  (773,055 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/03_rgi60_ArcticCanadaNorth_met_sampled.csv  (186,796 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/04_rgi60_ArcticCanadaSouth_met_sampled.csv  (304,015 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/05_rgi60_GreenlandPeriphery_met_sampled.csv  (830,701 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/06_rgi60_Iceland_met_sampled.csv  (23,288 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/07_rgi60_Svalbard_met_sampled.csv  (66,215 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/08_rgi60_Scandinavia_met_sampled.csv  (140,097 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/09_rgi60_RussianArctic_met_sampled.csv  (43,829 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/10_rgi60_NorthAsia_met_sampled.csv  (211,191 rows)
Wrote pergla_monthlymetw5e5_sampled_csv/11_rgi60_CentralEuro